In [ ]:
import torch

# Install the transformers library quietly
!pip install -q transformers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Determine if CUDA (GPU) is available, otherwise use CPU
torch_device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the tokenizer for the 'gpt2' model
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Add the EOS token as PAD token to avoid warnings when generating text
# Load the 'gpt2' model and move it to the specified device (CPU/GPU)
model = AutoModelForCausalLM.from_pretrained("gpt2", pad_token_id=tokenizer.eos_token_id).to(torch_device)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# Define the input prompt for text generation
prompt = "The future of artificial intelligence"

# Tokenize the prompt and convert it to PyTorch tensors, then move to the device
inputs = tokenizer(prompt, return_tensors="pt").to(torch_device)

# Print the tokenized inputs
print(inputs)
# Print the shape of the input IDs tensor
print(inputs["input_ids"].shape)
# Convert token IDs back to human-readable tokens and print them
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

{'input_ids': tensor([[  464,  2003,   286, 11666,  4430]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}
torch.Size([1, 5])
['The', 'Ġfuture', 'Ġof', 'Ġartificial', 'Ġintelligence']


In [ ]:
# Disable gradient calculations for inference to save memory and speed up computation
with torch.no_grad():
    # Pass the tokenized inputs to the model to get outputs
    outputs = model(**inputs)

# Print the shape of the output logits (raw prediction scores)
print(outputs.logits.shape)
# Extract the logits for the last token position
next_token_logits = outputs.logits[:, -1, :] #[batchsize:seqlen:vocabulary]

# Print the shape of the next token logits
print(next_token_logits.shape)

torch.Size([1, 5, 50257])
torch.Size([1, 50257])


In [ ]:
# Convert the logits for the next token into probabilities using softmax
next_token_probs = torch.softmax(next_token_logits, dim=-1)

# Print the shape of the probabilities tensor
print(next_token_probs.shape)
# Print the sum of probabilities to verify they sum to 1 (as expected for a probability distribution)
print(next_token_probs.sum())

torch.Size([1, 50257])
tensor(1.0000)


In [ ]:
# Get the top 10 probabilities and their corresponding token indices
top_probs, top_indices = torch.topk(next_token_probs, k=10)

# Iterate through the top probabilities and indices
for prob, idx in zip(top_probs[0], top_indices[0]):
    # Decode the token ID back to its string representation
    token = tokenizer.decode([idx])
    # Print the token and its probability, formatted to 4 decimal places
    print(f"{token!r}: {prob.item():.4f}")
# Print the full tensors of top probabilities and indices
print(top_probs,top_indices)

' is': 0.2891
',': 0.0640
' and': 0.0626
' will': 0.0588
' in': 0.0503
'\n': 0.0328
' may': 0.0295
' has': 0.0294
'.': 0.0207
':': 0.0203
tensor([[0.2891, 0.0640, 0.0626, 0.0588, 0.0503, 0.0328, 0.0295, 0.0294, 0.0207,
         0.0203]]) tensor([[318,  11, 290, 481, 287, 198, 743, 468,  13,  25]])


# **Greedy Search**

In [ ]:
# Initialize input_ids with the tokenized prompt
input_ids = inputs["input_ids"]

# Loop 10 times to generate 10 new tokens using greedy search
for _ in range(10):
    # Disable gradient calculations for inference
    with torch.no_grad():
        # Pass the current input_ids to the model
        outputs = model(input_ids=input_ids)

    # Get logits for the last token position in the sequence
    next_token_logits = outputs.logits[:, -1, :]

    # Convert logits to probabilities using softmax
    next_token_probs = torch.softmax(next_token_logits, dim=-1)

    # Greedy approach: choose the token with the highest probability
    next_token_id = torch.argmax(next_token_probs, dim=-1, keepdim=True)

    # Add the newly selected token to the input sequence for the next iteration
    input_ids = torch.cat([input_ids, next_token_id], dim=-1)

# Decode the complete generated sequence of tokens and print it
print(tokenizer.decode(input_ids[0]))

The future of artificial intelligence is uncertain.

"We're not sure


In [ ]:
# Initialize input_ids with the tokenized prompt
input_ids = inputs["input_ids"]

# Loop 10 times to generate 10 new tokens
for step in range(10):
    # Disable gradient calculations for inference
    with torch.no_grad():
        # Pass the current input_ids to the model
        outputs = model(input_ids=input_ids)

    # Get logits for the last token position
    next_token_logits = outputs.logits[:, -1, :]
    # Convert logits to probabilities using softmax
    next_token_probs = torch.softmax(next_token_logits, dim=-1)

    # Select the token with the highest probability (greedy approach)
    next_token_id = torch.argmax(
        next_token_probs,
        dim=-1,
        keepdim=True
    )

    # Decode the selected token ID to its string representation
    token = tokenizer.decode(next_token_id[0])
    # Get the probability of the selected token
    probability = next_token_probs[0, next_token_id.item()].item()

    # Print the step, the generated token, and its probability
    print(
        f"Step {step + 1}: "
        f"{token!r} → {probability:.4f}"
    )

    # Concatenate the new token to the input sequence for the next iteration
    input_ids = torch.cat(
        [input_ids, next_token_id],
        dim=-1
    )

Step 1: ' is' → 0.2891
Step 2: ' uncertain' → 0.0766
Step 3: '.' → 0.3799
Step 4: '\n' → 0.0756
Step 5: '\n' → 0.9994
Step 6: '"' → 0.0979
Step 7: 'We' → 0.1282
Step 8: "'re" → 0.2063
Step 9: ' not' → 0.1198
Step 10: ' sure' → 0.1472


# **Beam Search**

In [ ]:
beam_output=model.generate(**inputs,max_new_tokens=40,num_beams=5,early_stopping=True,no_repeat_ngram_size=2,num_return_sequences=5)
print("Output:\n" + 100 * '-')
print(tokenizer.decode(beam_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
The future of artificial intelligence is in the hands of the next generation of scientists and engineers.

"It's a very exciting time to be a part of it," he said. "I think we're going to see


# **Top-k sampling**

In [ ]:
from transformers import set_seed

# set seed to reproduce results. Feel free to change the seed though to get different results
set_seed(42)

# set top_k to 50
sample_output = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    top_k=50
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
The future of artificial intelligence is bound to be bright.

A new report from the Center for Computer and Information Sciences (CCIS), a nonprofit think tank that advocates for an open, open and transparent system of intellectual property


# **Top-p sampling**

In [ ]:
nulecus_output=model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    top_k=0,
    top_p=0.95
)
print("Output:\n"+100*"--")
print(tokenizer.decode(nulecus_output[0],skip_special_tokens=True))

Output:
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
The future of artificial intelligence is still quite far away, but as his PhD students envisions, "four years down the road, four years with the efficacy of machine learning or computational tools, four years with confidence in simple,


# **Combination(Top-k and Top-k) sampling**

In [ ]:
nulecus_output=model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    top_k=50,
    top_p=0.95
)
print("Output:\n"+100*"--")
print(tokenizer.decode(nulecus_output[0],skip_special_tokens=True))

Output:
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
The future of artificial intelligence is likely to be one of cooperation with industry, cooperation with government, and cooperation in social and cultural change, according to a report by a group that represents a lot of the big tech investors.

